# 8. Model Comparison


## Load the derived residual series

This notebook uses the residual series created by `02_decomposition.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

residual_series = pd.read_csv(
    "data/residual_series.csv",
    parse_dates=["date"],
    index_col="date"
)["residual"].dropna()

print(f"Loaded {len(residual_series)} residual observations.")


## Evaluation metrics

- **MAE (Mean Absolute Error):** average absolute difference between predicted and observed values.
- **RMSE (Root Mean Squared Error):** square-rooted average of squared prediction errors; larger errors receive more influence.

### Analogy
If a weather app predicts temperature, MAE asks how many degrees away the predictions were on average. RMSE gives extra influence to unusually large mistakes. The final numerical outputs come from the executed notebook.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.regression.linear_model import yule_walker
from sklearn.metrics import mean_absolute_error, mean_squared_error

# === Input ===
y = residual_series.values
index = residual_series.index

# === Utility: MAD for robust estimation ===
def mad(x):
    return np.median(np.abs(x - np.median(x))) + 1e-8

# === NCV functions ===
def ncv_autocorr(series, lag):
    if lag == 0:
        return 1.0
    x = series[:-lag]
    y = series[lag:]
    return np.median(x * y) / (mad(x) * mad(y))

def ncv_yule_walker(series, L):
    r = [ncv_autocorr(series, h) for h in range(L + 1)]
    R = np.array([[r[abs(i - j)] for j in range(L)] for i in range(L)])
    r_vec = np.array(r[1:L+1])
    try:
        phi = np.linalg.solve(R, r_vec)
    except np.linalg.LinAlgError:
        phi = np.zeros(L)
    return phi

# === FLoC functions ===
def floc(x, y, tau=0.5):
    return np.quantile(y * x, tau) / (np.quantile(x ** 2, tau) + 1e-8)

def floc_ar_coeffs(series, L, tau=0.5):
    coeffs = []
    for lag in range(1, L + 1):
        x = series[:-lag]
        y_ = series[lag:]
        coeff = floc(x, y_, tau)
        coeffs.append(coeff)
    return np.array(coeffs)

# === General Fitting Functions ===
def fit_par_model(y, p, L, dep_type="classical"):
    n = len(y)
    phi = {}
    fitted = np.zeros_like(y)

    for s in range(p):
        y_s = np.array([y[t] for t in range(p * L, n) if t % p == s])
        if len(y_s) <= L + 1:
            phi[s] = np.zeros(L)
            continue
        if dep_type == "classical":
            try:
                rho, _ = yule_walker(y_s, order=L)
                phi[s] = rho
            except:
                phi[s] = np.zeros(L)
        elif dep_type == "ncv":
            phi[s] = ncv_yule_walker(y_s, L)
        elif dep_type == "floc":
            phi[s] = floc_ar_coeffs(y_s, L)
        else:
            raise ValueError("Invalid dep_type")

    for t in range(p * L, n):
        s = t % p
        lag_vec = y[t - L:t][::-1]
        fitted[t] = np.dot(phi.get(s, np.zeros(L)), lag_vec)

    return phi, fitted

# === Error Function ===
def compute_errors(y_true, y_pred, model_name):
    start = np.argmax(y_pred != 0)
    y_true = y_true[start:]
    y_pred = y_pred[start:]
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{model_name}:\n  MAE  = {mae:.4f}\n  RMSE = {rmse:.4f}\n")
    return mae, rmse, y_pred

# === Set p and L manually or tune via AIC ===
p = 12  # example seasonality
L = 2   # example AR order

# === Fit Models ===
phi_classical, fitted_classical = fit_par_model(y, p, L, dep_type="classical")
phi_ncv,       fitted_ncv       = fit_par_model(y, p, L, dep_type="ncv")
phi_floc,      fitted_floc      = fit_par_model(y, p, L, dep_type="floc")

# === Compute Errors ===
mae_c, rmse_c, yhat_c = compute_errors(y, fitted_classical, "Classical Yule-Walker PAR")
mae_n, rmse_n, yhat_n = compute_errors(y, fitted_ncv, "NCV-based PAR")
mae_f, rmse_f, yhat_f = compute_errors(y, fitted_floc, "FLoC-based PAR")

# === Create Table ===
error_df = pd.DataFrame({
    "Model": ["Classical YW", "NCV", "FLoC"],
    "MAE": [mae_c, mae_n, mae_f],
    "RMSE": [rmse_c, rmse_n, rmse_f]
})
print("=== Error Comparison Table ===")
print(error_df)

# === Plot ===
# Determine the warm-up point for all models
start = max(p * L, np.argmax(fitted_classical != 0), np.argmax(fitted_ncv != 0), np.argmax(fitted_floc != 0))

plt.figure(figsize=(14, 5))
plt.plot(index[start:], y[start:], label="Actual Residuals", color='black', alpha=0.4)
plt.plot(index[start:], fitted_classical[start:], label="Classical YW", color='green', alpha=0.7)
plt.plot(index[start:], fitted_ncv[start:], label="NCV", color='purple', alpha=0.7)
plt.plot(index[start:], fitted_floc[start:], label="FLoC", color='orange', alpha=0.7)

plt.title("Comparison of PAR Model Predictions")
plt.xlabel("Time")
plt.ylabel("Residual Value")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


